# 数字广告投放 A/B 测试：购买转化效果分析

**Marketing Ad vs PSA A/B Test — Conversion Analysis**

本 Notebook 保留并整理原项目中已经完成的数据质量检查、分组转化率汇总和广告曝光频次分析，并补充可复现的双样本比例 Z 检验。数据来自 Kaggle 的匿名实验数据，未披露真实企业名称。

数据来源：[Marketing A/B Testing](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing)

## 1. 读取数据与质量检查

原始 CSV 保持不变。分析副本中删除无业务含义的 `Unnamed: 0` 列，并检查缺失值、重复用户、字段类型、实验分组、购买结果、曝光次数、星期和小时字段。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm


# 同时兼容从项目根目录或 notebooks/ 目录启动 Jupyter。
csv_candidates = [
    Path("data/marketing_AB.csv"),
    Path("../data/marketing_AB.csv"),
]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "未找到 marketing_AB.csv。请从 Kaggle 下载并放到项目的 data/ 目录。"
    )

# 读取原始数据，不修改原文件。
df_raw = pd.read_csv(csv_path)
print("数据行列数：", df_raw.shape)
print("\n前 5 行：")
print(df_raw.head())
print("\n字段类型：")
print(df_raw.dtypes)
print("\n各字段缺失值数量：")
print(df_raw.isna().sum())
print("\n重复用户 ID 数量：", df_raw["user id"].duplicated().sum())

# 检查实验分组与购买结果。
group = df_raw["test group"].astype("string").str.strip().str.lower()
converted = df_raw["converted"].astype("string").str.strip().str.lower()
print("\n实验分组：")
print(df_raw["test group"].value_counts(dropna=False))
print("\n购买结果：")
print(df_raw["converted"].value_counts(dropna=False))
print("异常实验分组数量：", (~group.isin(["ad", "psa"])).sum())
print("异常购买结果数量：", (~converted.isin(["true", "false"])).sum())

# 检查曝光次数、星期和小时字段。
ads = pd.to_numeric(df_raw["total ads"], errors="coerce")
invalid_ads = ads.isna() | (ads <= 0) | (ads % 1 != 0)
valid_days = {
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday",
}
days = df_raw["most ads day"].astype("string").str.strip().str.lower()
hours = pd.to_numeric(df_raw["most ads hour"], errors="coerce")
invalid_hours = hours.isna() | ~hours.between(0, 23) | (hours % 1 != 0)
print("异常曝光次数数量：", invalid_ads.sum())
print("异常星期数量：", (~days.isin(valid_days)).sum())
print("异常小时数量：", invalid_hours.sum())

# 仅在分析副本中删除无业务意义的导出索引列。
df = df_raw.drop(columns=["Unnamed: 0"]).copy()
print("\n分析数据行列数：", df.shape)

数据行列数： (588101, 7)

前 5 行：
   Unnamed: 0  user id test group  converted  total ads most ads day  \
0           0  1069124         ad      False        130       Monday   
1           1  1119715         ad      False         93      Tuesday   
2           2  1144181         ad      False         21      Tuesday   
3           3  1435133         ad      False        355      Tuesday   
4           4  1015700         ad      False        276       Friday   

   most ads hour  
0             20  
1             22  
2             18  
3             10  
4             14  

字段类型：
Unnamed: 0        int64
user id           int64
test group       object
converted          bool
total ads         int64
most ads day     object
most ads hour     int64
dtype: object

各字段缺失值数量：
Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

重复用户 ID 数量： 0



实验分组：
test group
ad     564577
psa     23524
Name: count, dtype: int64

购买结果：
converted
False    573258
True      14843
Name: count, dtype: int64
异常实验分组数量： 0
异常购买结果数量： 0
异常曝光次数数量： 0
异常星期数量： 0
异常小时数量： 0

分析数据行列数： (588101, 6)


质量检查未发现缺失值、重复用户、非法分组、非法购买结果或异常时间字段取值。原始数据为 588,101 行；清理后仅删除一列无业务意义的导出索引。

## 2. 购买转化率

使用 pandas 按实验组汇总去重用户数和购买人数，再计算购买转化率、Absolute Lift 与 Relative Lift。

In [2]:
# 按实验组汇总用户数和购买人数（保留原分析的 pandas 分组聚合逻辑）。
result = df.groupby("test group").agg(
    users=("user id", "nunique"),
    purchases=("converted", "sum"),
)
result["conversion_rate"] = result["purchases"] / result["users"]

ad_rate = result.loc["ad", "conversion_rate"]
psa_rate = result.loc["psa", "conversion_rate"]
absolute_lift = ad_rate - psa_rate
relative_lift = absolute_lift / psa_rate

display_result = result.copy()
display_result["conversion_rate"] = display_result["conversion_rate"].map(
    lambda value: f"{value:.4%}"
)
print(display_result)
print(f"\nAbsolute Lift: {absolute_lift:.4%}（{absolute_lift * 100:.4f} 个百分点）")
print(f"Relative Lift: {relative_lift:.2%}")

             users  purchases conversion_rate
test group                                   
ad          564577      14423         2.5547%
psa          23524        420         1.7854%

Absolute Lift: 0.7692%（0.7692 个百分点）
Relative Lift: 43.09%


## 3. 统计显著性检验

- 双侧双样本比例 Z 检验，$\alpha = 0.05$；
- 零假设 $H_0: p_{Ad} = p_{PSA}$；
- Z 统计量在零假设下采用 pooled standard error；
- 转化率差（Ad − PSA）的 95% 置信区间采用两组独立比例的 **unpooled Wald interval**。

原始 MDE 与预期分流比例未知，因此不把练习时填写的参数当作原始实验设计，也不对 SRM 作“已通过”的判断。

In [3]:
ad_users = int(result.loc["ad", "users"])
ad_purchases = int(result.loc["ad", "purchases"])
psa_users = int(result.loc["psa", "users"])
psa_purchases = int(result.loc["psa", "purchases"])

# 双样本比例 Z 检验：H0 下使用 pooled standard error。
pooled_rate = (ad_purchases + psa_purchases) / (ad_users + psa_users)
pooled_se = np.sqrt(
    pooled_rate * (1 - pooled_rate) * (1 / ad_users + 1 / psa_users)
)
z_stat = absolute_lift / pooled_se
p_value = 2 * norm.sf(abs(z_stat))

# 两个独立比例之差的 unpooled Wald 95% CI。
unpooled_se = np.sqrt(
    ad_rate * (1 - ad_rate) / ad_users
    + psa_rate * (1 - psa_rate) / psa_users
)
critical_value = norm.ppf(0.975)
ci_low = absolute_lift - critical_value * unpooled_se
ci_high = absolute_lift + critical_value * unpooled_se

print(f"Z statistic: {z_stat:.4f}")
print(f"Two-sided p-value: {p_value:.4e}")
print(
    "95% CI for Ad - PSA: "
    f"[{ci_low:.6f}, {ci_high:.6f}] "
    f"= [{ci_low * 100:.4f}, {ci_high * 100:.4f}] 个百分点"
)
print("统计结论：", "拒绝 H0，转化率差异具有统计显著性。" if p_value < 0.05 else "未拒绝 H0。")

Z statistic: 7.3701
Two-sided p-value: 1.7053e-13
95% CI for Ad - PSA: [0.005951, 0.009434] = [0.5951, 0.9434] 个百分点
统计结论： 拒绝 H0，转化率差异具有统计显著性。


结果与 [A/B Testing Decision Toolkit](https://ab-testing-decision-toolkit.onrender.com/) 的输出一致（工具 [GitHub 仓库](https://github.com/alexxiaohc/ab-testing-decision-toolkit)）。统计显著不等于业务上一定盈利，仍需要成本与利润数据。

## 4. Ad 组广告曝光频次探索

先复现 Ad 组曝光次数的描述性统计，再按 1–5、6–10、11–20、21–50、51–100、101+ 六个区间汇总用户数、购买人数与转化率。

In [4]:
# 只分析广告组用户的曝光次数（保留原分析代码逻辑）。
ad_ads = df.loc[df["test group"] == "ad", "total ads"]
print(ad_ads.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

count    564577.000000
mean         24.823365
std          43.750456
min           1.000000
25%           4.000000
50%          13.000000
75%          27.000000
90%          56.000000
95%          87.000000
99%         201.000000
max        2065.000000
Name: total ads, dtype: float64


In [5]:
ad_df = df[df["test group"] == "ad"].copy()
bins = [0, 5, 10, 20, 50, 100, float("inf")]
labels = ["1–5", "6–10", "11–20", "21–50", "51–100", "101+"]

ad_df["ads_group"] = pd.cut(
    ad_df["total ads"],
    bins=bins,
    labels=labels,
)

ads_result = ad_df.groupby("ads_group", observed=True).agg(
    users=("user id", "nunique"),
    purchases=("converted", "sum"),
)
ads_result["conversion_rate"] = ads_result["purchases"] / ads_result["users"]

ads_display = ads_result.copy()
ads_display["conversion_rate"] = ads_display["conversion_rate"].map(
    lambda value: f"{value:.2%}"
)
print(ads_display)

            users  purchases conversion_rate
ads_group                                   
1–5        169962        427           0.25%
6–10        79537        386           0.49%
11–20      123334       1036           0.84%
21–50      125541       3660           2.92%
51–100      44149       5135          11.63%
101+        22054       3779          17.14%


高曝光区间的购买转化率更高，但 `total ads` 并未独立随机分配，且可能受到用户活跃度、购买意愿或投放策略等因素影响。因此这里仅报告相关性，不能解释为“增加曝光次数导致购买转化率提高”。

## 5. 增量转化估算与业务结论

在随机分组有效且两组可比的前提下，用观察到的转化率差乘以 Ad 组用户数，估算 Ad 相对 PSA 带来的增量购买人数。该估算不包含广告成本、订单金额或贡献利润。

In [6]:
incremental_purchases = absolute_lift * ad_users
print(f"估算增量购买人数：{incremental_purchases:,.2f}，约 {round(incremental_purchases):,} 人")

估算增量购买人数：4,342.98，约 4,343 人


### 核心结论

1. Ad 组购买转化率显著高于 PSA 对照组。
2. 因原始随机分流方案未知，因果结论需要以随机分组有效、两组可比为前提。
3. 数据缺少广告费用、订单金额和贡献利润，不能计算真实 ROI。
4. 当前无法判断扩大广告投放是否提高利润，因此不直接建议扩大预算。
5. 后续应补充成本和贡献利润数据，并评估边际 ROI。